# YZ50 — Week 9: Building the GPT Tokenizer

- **Unicode** assigns every character an integer **code point**, while **UTF-8** stores each code point as a sequence of **1 to 4 bytes**. Turkish letters such as `ç, ğ, ı, ö, ş, ü` need two bytes each.

- A **byte-level vocabulary** starts from the 256 possible byte values, so any string can be represented without an unknown-character token.

- **Byte Pair Encoding (BPE)** repeatedly merges the most frequent adjacent pair of tokens into a new token, compressing frequent byte sequences into single IDs.

- **Encoding** applies the learned merges in their training order; **decoding** concatenates the token bytes and interprets them as UTF-8. Valid text must survive the round trip exactly.

- **Regex pre-tokenization** with the GPT-4 `cl100k_base` pattern first splits text into words, numbers, punctuation, and whitespace. BPE then merges only inside each chunk.

- **Turkish vs. English tokenization** shows how agglutinative suffixes make Turkish words long, and how this changes token counts under GPT-4's tokenizer and under a tokenizer trained on Turkish.

In this week, I follow the tokenization lecture from raw UTF-8 bytes to a GPT-4-style tokenizer. I train a basic BPE tokenizer and a regex-aware BPE tokenizer on the first 90% of my Week 8 Turkish corpus with `vocab_size = 512`, check their encode/decode round trips with assertions, compare their merges and compression on the held-out 10%, and measure Turkish and English token counts with `cl100k_base`.

In [1]:
from pathlib import Path
import time
import unicodedata
from collections import Counter
import regex as re
import tiktoken
from IPython.display import display, Markdown

data_dir = Path.cwd()
if not (data_dir / 'turkish_input.txt').exists():
    data_dir = data_dir / 'week-9'
print('regex:', re.__version__, 'tiktoken:', tiktoken.__version__)

regex: 2026.9.29 tiktoken: 0.14.0


## Tokenization

Tokenization is the step between raw text and the integer IDs a language model actually sees. Many LLM oddities, such as poor spelling, trouble reversing strings, weak arithmetic, and worse performance outside English, trace back to how text is cut into tokens.

My Week 8 tokenizer was **character-level**: 92 Turkish characters, one ID each. GPT models use **byte-level BPE**, which I briefly described in Week 7.

The tokenizer is a **separate module** from the LLM. It has its own training text, learns its vocabulary with BPE, and then translates between text and token sequences. The LLM only ever sees the tokens.

Good tokenization web app: [https://tiktokenizer.vercel.app](https://tiktokenizer.vercel.app)

# 1. Metni byte'lara çevir. Türkçe harfler içeren bir cümle seç (ç, ğ, ı, ö, ş, ü). Her harfin Unicode kod noktasını ve UTF-8 byte'larını yazdır. Hangi harfler 1 byte, hangileri 2 byte tutuyor? Kod noktalarını doğrudan vocab olarak kullanmak neden kötü bir fikir, kendi cümlelerinle yaz.

## 1. Unicode Code Points and UTF-8 Bytes

A Python string is a sequence of **Unicode code points**. `ord()` returns the code point of one character. `.encode("utf-8")` returns the **UTF-8 bytes**, integers in the range 0..255.

The next three cells reproduce the lecture's Korean example.

In [2]:
"안녕하세요 👋 (hello in Korean!)"

'안녕하세요 👋 (hello in Korean!)'

In [3]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

In [4]:
list("안녕하세요 👋 (hello in Korean!)".encode("utf-8"))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

### A Turkish Sentence

*Pijamalı hasta yağız şoföre çabucak güvendi* is a Turkish pangram that contains all six required letters. I add `İzmir`, `Işık`, and an emoji to compare `I`/`İ` and a four-byte character.

In [5]:
sentence = "Pijamalı hasta yağız şoföre çabucak güvendi. İzmir'de Işık 👋"

print(f"{'char':6} {'code point':10} {'decimal':>7}  {'UTF-8 bytes':18} {'ints':22} {'len':>3}  name")
for ch in dict.fromkeys(sentence): # unique characters, in order of appearance
    b = ch.encode("utf-8")
    print(f"{ch!r:6} U+{ord(ch):04X}     {ord(ch):>7}  {b.hex(' '):18} {str(list(b)):22} {len(b):>3}  {unicodedata.name(ch)}")

char   code point decimal  UTF-8 bytes        ints                   len  name
'P'    U+0050          80  50                 [80]                     1  LATIN CAPITAL LETTER P
'i'    U+0069         105  69                 [105]                    1  LATIN SMALL LETTER I
'j'    U+006A         106  6a                 [106]                    1  LATIN SMALL LETTER J
'a'    U+0061          97  61                 [97]                     1  LATIN SMALL LETTER A
'm'    U+006D         109  6d                 [109]                    1  LATIN SMALL LETTER M
'l'    U+006C         108  6c                 [108]                    1  LATIN SMALL LETTER L
'ı'    U+0131         305  c4 b1              [196, 177]               2  LATIN SMALL LETTER DOTLESS I
' '    U+0020          32  20                 [32]                     1  SPACE
'h'    U+0068         104  68                 [104]                    1  LATIN SMALL LETTER H
's'    U+0073         115  73                 [115]                    

In [6]:
by_length = {}
for ch in dict.fromkeys(sentence):
    by_length.setdefault(len(ch.encode("utf-8")), []).append(ch)
for n_bytes, chs in sorted(by_length.items()):
    print(n_bytes, 'byte(s):', ' '.join(repr(ch) for ch in chs))

for ch in 'çğıöşü':
    assert ch in sentence
    assert len(ch.encode("utf-8")) == 2
assert len('İ'.encode("utf-8")) == 2 and len('I'.encode("utf-8")) == 1
assert len('👋'.encode("utf-8")) == 4

print('characters:', len(sentence), 'UTF-8 bytes:', len(sentence.encode("utf-8")))
print("'İ'.lower() =", repr('İ'.lower()), 'has', len('İ'.lower()), 'code points') # one reason not to lowercase

1 byte(s): 'P' 'i' 'j' 'a' 'm' 'l' ' ' 'h' 's' 't' 'y' 'z' 'o' 'f' 'r' 'e' 'b' 'u' 'c' 'k' 'g' 'v' 'n' 'd' '.' "'" 'I'
2 byte(s): 'ı' 'ğ' 'ş' 'ö' 'ç' 'ü' 'İ'
4 byte(s): '👋'
characters: 60 UTF-8 bytes: 73
'İ'.lower() = 'i̇' has 2 code points


### Why 1, 2, or 4 Bytes?

UTF-8 is a **variable-length encoding**. The leading bits of the first byte tell the decoder how many bytes belong to the character:

| Code point range | UTF-8 bytes | Byte pattern | Examples |
|---|---:|---|---|
| U+0000 – U+007F | 1 | `0xxxxxxx` | ASCII letters, `I`, space, `'` |
| U+0080 – U+07FF | 2 | `110xxxxx 10xxxxxx` | `ç ğ ı ö ş ü İ` |
| U+0800 – U+FFFF | 3 | `1110xxxx 10xxxxxx 10xxxxxx` | Korean syllables |
| U+10000 – U+10FFFF | 4 | `11110xxx 10xxxxxx 10xxxxxx 10xxxxxx` | `👋` |

- **ASCII** code points are below 128 and fit in the 7 free bits of one byte, so UTF-8 stores them unchanged.
- The **Turkish letters** have code points from 231 (`ç`) to 351 (`ş`). They do not fit in 7 bits, so they use the two-byte form with 11 free bits.
- **`👋`** is U+1F44B (128,075), above U+FFFF, so it needs the four-byte form.

A **code point** is the character's number in the Unicode standard; **UTF-8 bytes** are one way of storing that number. `ğ` is the single code point 287 but the two bytes `[196, 159]`. Bytes of the form `10xxxxxx` are **continuation bytes**: they cannot start a character on their own.

### Why Not Use Code Points Directly as the Vocabulary?

Code points can be tokenizer units; SentencePiece, for example, runs BPE over code points. The problem is using the raw code point **value** as the token ID of a model:

- The Unicode code space has **1,114,112** values. Token 351 (`ş`) or 128,075 (`👋`) would need an embedding row and an output logit in a table of that size, and almost all of those rows would never be trained.
- Remapping only the characters seen in the corpus, as in my Week 8 tokenizer, gives a dense vocabulary, but it is **closed**: a character absent from the training text has no ID.
- Either way, every character is still one token, so there is **no compression**.

UTF-8 bytes give a small, open base vocabulary: **256 IDs** can encode any string. The cost is longer sequences, since every Turkish letter becomes two tokens. BPE then compresses frequent byte sequences back into single tokens.

# 2. BPE'yi sıfırdan yaz ve kendi Türkçe metninde eğit. Geçen hafta derlediğin Türkçe metni kullan. Ardışık ikilileri say, en sık geçeni yeni bir token'la birleştir, istediğin vocab boyutuna gelene kadar tekrarla. İlk 20 birleştirmeyi metin olarak yazdır: hangi harf grupları önce birleşiyor? Sıkıştırma oranını (byte sayısı / token sayısı) yaz.

## 2. Byte Pair Encoding from Scratch

BPE starts from the byte sequence and repeats one step:

**count adjacent pairs → take the most frequent pair → replace it with a new token ID → repeat**

The first new token gets ID 256, the next 257, and so on. `merges` stores every `(p0, p1) → idx` rule in the order it was learned, so `vocab_size = 256 + number of merges`. With `vocab_size = 512`, the tokenizer learns up to 256 merges.

**Ties:** `max(stats, key=stats.get)` returns the first pair with the highest count in dictionary order, which is the order in which pairs were first encountered in the text. For a fixed text the merges are deterministic.

**Data:** I read the corpus in text mode with UTF-8, as in Week 8. The file has Windows `\r\n` line endings, which text mode translates to `\n`. Nothing else changes: no lowercasing or cleaning, so soft hyphens, `’`, and `«»` remain. As in Week 8, the first 90% of the characters is `train_text` and the last 10% is `val_text`. Merges are learned only from `train_text`.

In [7]:
# read it in to inspect it
with open(data_dir / 'turkish_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

raw_bytes = (data_dir / 'turkish_input.txt').read_bytes()
print('file bytes:', len(raw_bytes), '| CRLF line endings:', raw_bytes.count(b'\r\n'))
print('length of dataset in characters:', len(text))
print('length of dataset in UTF-8 bytes:', len(text.encode('utf-8')))
assert '\r' not in text
assert text.encode('utf-8') == raw_bytes.replace(b'\r\n', b'\n') # only the line endings changed

# first 90% will be train, rest val
n = int(0.9*len(text))
train_text = text[:n]
val_text = text[n:]
print('train characters:', len(train_text), '| val characters:', len(val_text))

file bytes: 339481 | CRLF line endings: 1478
length of dataset in characters: 302802
length of dataset in UTF-8 bytes: 338003
train characters: 272521 | val characters: 30281


In [8]:
tokens = train_text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(train_text[:300])
print("length:", len(train_text))
print('---')
print(tokens[:60])
print("length:", len(tokens))

---
 Simdiye kadar tesadüf ettiğim insanlardan bir tanesi benim üzerimde belki en büyük tesiri yapmıştır. Aradan aylar geçtiği halde bir türlü bu tesirden kurtulamadım. Ne zaman kendimle başbaşa kalsam, Raif Efendinin saf yüzü, biraz dünyadan uzak, buna rağmen bir insana tesadüf ettikleri zaman şaşkınca
length: 272521
---
[226, 128, 131, 83, 105, 109, 100, 105, 121, 101, 32, 107, 97, 100, 97, 114, 32, 116, 101, 115, 97, 100, 195, 188, 102, 32, 101, 116, 116, 105, 196, 159, 105, 109, 32, 105, 110, 115, 97, 110, 108, 97, 114, 100, 97, 110, 32, 98, 105, 114, 32, 116, 97, 110, 101, 115, 105, 32, 98, 101]
length: 304535


In [9]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def render(b):
    # bytes -> readable text; incomplete UTF-8 stays visible as \xNN instead of being hidden
    s = b.decode('utf-8', errors='backslashreplace')
    s = ''.join(ch if ch.isprintable() else repr(ch)[1:-1] for ch in s)
    q = '"' if "'" in s else "'"
    return q + s + q

stats = get_stats(tokens)
print('distinct pairs:', len(stats))
for count, pair in sorted(((v,k) for k,v in stats.items()), reverse=True)[:10]:
    print(pair, render(bytes(pair)), count)

distinct pairs: 1230
(196, 177) 'ı' 10406
(32, 98) ' b' 5084
(110, 32) 'n ' 4827
(195, 188) 'ü' 4425
(101, 32) 'e ' 4310
(97, 114) 'ar' 3890
(197, 159) 'ş' 3700
(97, 32) 'a ' 3574
(101, 114) 'er' 3310
(114, 32) 'r ' 3238


In [10]:
top_pair = max(stats, key=stats.get)
print(top_pair, render(bytes(top_pair)), stats[top_pair])

(196, 177) 'ı' 10406


In [11]:
def merge(ids, pair, idx):
    # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
    newids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position AND the pair matches, replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))
assert merge([5, 6, 6, 7, 9, 1], (6, 7), 99) == [5, 6, 99, 9, 1]
assert merge([1, 1, 1], (1, 1), 256) == [256, 1] # overlapping pairs merge left to right
assert merge([], (1, 2), 256) == [] and merge([1], (1, 2), 256) == [1]

tokens2 = merge(tokens, top_pair, 256)
print("length:", len(tokens2))

[5, 6, 99, 9, 1]
length: 294129


### Train the Basic Tokenizer

The loop stops early if fewer than two tokens remain, so `vocab_size` is an upper bound. `merge_counts` records the pair frequency at the moment each pair was merged.

In [12]:
vocab_size = 512 # the desired final vocabulary size
num_merges = vocab_size - 256
assert num_merges >= 0
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
merge_counts = {} # (int, int) -> pair count when it was merged
start = time.perf_counter()
for i in range(num_merges):
    stats = get_stats(ids)
    if not stats:
        break # fewer than two tokens left, nothing to merge
    pair = max(stats, key=stats.get) # ties: the first pair encountered wins
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    merge_counts[pair] = stats[pair]
basic_seconds = time.perf_counter() - start
print('learned merges:', len(merges), '| training seconds:', round(basic_seconds, 2))

learned merges: 256 | training seconds: 10.68


In [13]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
assert len(vocab) == 256 + len(merges) == vocab_size

def show_merges(merges, merge_counts, vocab, k=20):
    print(f"{'rank':>4} | {'pair':>10} | {'left':>8} + {'right':<8} -> {'new token':<16} | count")
    for rank, ((p0, p1), idx) in enumerate(list(merges.items())[:k], start=1):
        print(f"{rank:>4} | {str((p0, p1)):>10} | {render(vocab[p0]):>8} + {render(vocab[p1]):<8} -> {idx} {render(vocab[idx]):<12} | {merge_counts[(p0, p1)]}")

show_merges(merges, merge_counts, vocab)

rank |       pair |     left + right    -> new token        | count
   1 | (196, 177) |   '\xc4' + '\xb1'   -> 256 'ı'          | 10406
   2 |   (32, 98) |      ' ' + 'b'      -> 257 ' b'         | 5084
   3 | (195, 188) |   '\xc3' + '\xbc'   -> 258 'ü'          | 4425
   4 |  (97, 114) |      'a' + 'r'      -> 259 'ar'         | 3890
   5 |  (110, 32) |      'n' + ' '      -> 260 'n '         | 3866
   6 | (197, 159) |   '\xc5' + '\x9f'   -> 261 'ş'          | 3700
   7 |  (101, 32) |      'e' + ' '      -> 262 'e '         | 3496
   8 | (101, 114) |      'e' + 'r'      -> 263 'er'         | 3310
   9 |   (97, 32) |      'a' + ' '      -> 264 'a '         | 2942
  10 | (105, 114) |      'i' + 'r'      -> 265 'ir'         | 2813
  11 |   (46, 32) |      '.' + ' '      -> 266 '. '         | 2602
  12 | (196, 159) |   '\xc4' + '\x9f'   -> 267 'ğ'          | 2540
  13 |  (105, 32) |      'i' + ' '      -> 268 'i '         | 2500
  14 | (195, 167) |   '\xc3' + '\xa7'   -> 269 'ç'          

In [14]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

print('\nlongest learned tokens:')
for idx in sorted(range(256, vocab_size), key=lambda i: -len(vocab[i]))[:12]:
    print(idx, render(vocab[idx]))

tokens length: 304535
ids length: 137020
compression ratio: 2.22X

longest learned tokens:
437 ':\n\u2003\u2003«'
364 '\n\u2003\u2003«'
401 '.\n\u2003\u2003'
470 'yordum. '
310 '\n\u2003\u2003'
428 'yordu. '
468 'ığım'
388 'ını '
452 'için '
459 'ında '
469 'i bir '
481 'a bir '


### Which Byte Groups Merge First?

The first 20 merges fall into four groups:

- **Turkish two-byte letters:** `ı` is the very first merge (10,406 pairs), followed by `ü` (3rd), `ş` (6th), `ğ` (12th), and `ç` (14th). These letters are frequent, and UTF-8 stores each of them as two bytes, so their byte pairs are among the most frequent pairs of the text. `ö` is rarer and is not among the first 20.

  BPE does not know where a character begins or ends; it only counts adjacent byte pairs. Here the early merges happen to rebuild whole characters, but nothing guarantees this. A learned token can hold an incomplete UTF-8 sequence (Task 3 finds `\xe2\x80`, the first two bytes of both `’` and the em space), and one character can be spread over several tokens. Valid text still round-trips, because decoding joins all token bytes in order before interpreting them as UTF-8.
- **Letter pairs:** `ar`, `er`, `ir`, `ak`, `an`, `in`, `de`, `di`. These pairs can belong to frequent Turkish endings such as *-lar/-ler*, *-ir*, *-ın/-in*, *-da/-de*, and *-dı/-di*.
- **A letter followed by a space:** `n `, `e `, `a `, `i `. Many Turkish words end in these letters, so the end of the word is merged together with the next space.
- **Spaces and punctuation:** `' b'` (the start of ` bir` and ` bu`, two of the most frequent words), `'. '`, and `', '`.

Later merges build longer pieces: `ığım`, `ını `, `ında `, `için `, `yordu. ` and `yordum. ` (the past continuous ending with a full stop), `a bir `, and `\n\u2003\u2003«`, because paragraphs in this file start with two em spaces (U+2003) and dialogue with `«`. Several of these tokens cross word boundaries; Task 4 addresses this.

With 256 merges, the 304,535 training bytes become 137,020 tokens: a **compression ratio of 2.22X**.

# 3. encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden?

## 3. Decoding and Encoding

### Decoding

Given a sequence of integers in the range [0, vocab_size), what is the text? `decode` looks up the bytes of each token in `vocab`, concatenates them, and decodes the result as UTF-8.

Not every token sequence is valid UTF-8. `[128]` is a lone continuation byte, and a learned token can stop in the middle of a multi-byte character. With `errors="replace"`, invalid bytes become the replacement character `�` (U+FFFD) instead of raising `UnicodeDecodeError`. A language model can sample any token sequence, so decoding should always return something displayable; strict decoding would crash on such output. The `�` marks missing information: it is not a character of the original text.

In [15]:
def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

print(decode([128]))

�


In [16]:
def is_valid_utf8(b):
    try:
        b.decode('utf-8')
        return True
    except UnicodeDecodeError:
        return False

# learned tokens whose bytes are not complete UTF-8 on their own
partial = [idx for idx in range(256, vocab_size) if not is_valid_utf8(vocab[idx])]
print('learned tokens that are incomplete UTF-8:', len(partial), ' '.join(render(vocab[i]) for i in partial[:8]))

examples = [('lone continuation byte', [128]),
            ('first byte of ı', list('ı'.encode('utf-8'))[:1]),
            ('complete ı', list('ı'.encode('utf-8'))),
            ('first 3 bytes of 👋', list('👋'.encode('utf-8'))[:3])]
if partial:
    examples.append(('learned token ' + str(partial[0]), [partial[0]]))
for name, seq in examples:
    raw = b"".join(vocab[idx] for idx in seq)
    try:
        strict = repr(raw.decode('utf-8'))
    except UnicodeDecodeError as e:
        strict = 'UnicodeDecodeError (' + e.reason + ')'
    print(f"{name:24} {str(raw):18} strict: {strict:45} replace: {decode(seq)!r}")

learned tokens that are incomplete UTF-8: 1 '\xe2\x80'
lone continuation byte   b'\x80'            strict: UnicodeDecodeError (invalid start byte)       replace: '�'
first byte of ı          b'\xc4'            strict: UnicodeDecodeError (unexpected end of data)   replace: '�'
complete ı               b'\xc4\xb1'        strict: 'ı'                                           replace: 'ı'
first 3 bytes of 👋       b'\xf0\x9f\x91'    strict: UnicodeDecodeError (unexpected end of data)   replace: '�'
learned token 278        b'\xe2\x80'        strict: UnicodeDecodeError (unexpected end of data)   replace: '�'


### Encoding

The other way around: given a string, what are the tokens? `encode` starts from the UTF-8 bytes and repeatedly applies the **learned merge with the lowest index** present in the sequence, not the most frequent pair of the new text. Merges must follow their training priority because later merges are built from tokens that earlier merges created.

**Two different guarantees:**

- For any valid Python string, `decode(encode(s)) == s` exactly, because encoding only regroups the string's UTF-8 bytes.
- For an arbitrary list of IDs, `decode` always returns a string, but `encode(decode(ids))` need not give back `ids`, and replaced bytes are lost.

In [17]:
def encode(text):
    # given a string, return list of integers (the tokens)
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode(""))
print(encode("a"))
print(decode(encode("hello world")))

[]
[97]
hello world


In [18]:
samples = [
    "",
    "hello world",
    sentence,
    "Pijamalı hasta yağız şoföre çabucak güvendi.",
    "İstanbul'da çay içtik ☕ ve çok güldük 😄",
    "ıiİI: IĞDIR, ılık, İzmir, Işık, iğne",
    "Birinci satır\nİkinci satır\n\nüçüncü satır",
    "kar\xadşılaşmamız", # soft hyphen, as in the corpus
    "👨‍👩‍👧 bir aile",
]
for s in samples:
    ids_s = encode(s)
    assert decode(ids_s) == s
    print(f"chars {len(s):3d} | bytes {len(s.encode('utf-8')):3d} | tokens {len(ids_s):3d} | {s!r}")
print('Round-trip checks passed.')

chars   0 | bytes   0 | tokens   0 | ''
chars  11 | bytes  11 | tokens   9 | 'hello world'
chars  60 | bytes  73 | tokens  43 | "Pijamalı hasta yağız şoföre çabucak güvendi. İzmir'de Işık 👋"
chars  44 | bytes  51 | tokens  30 | 'Pijamalı hasta yağız şoföre çabucak güvendi.'
chars  39 | bytes  50 | tokens  29 | "İstanbul'da çay içtik ☕ ve çok güldük 😄"
chars  36 | bytes  45 | tokens  29 | 'ıiİI: IĞDIR, ılık, İzmir, Işık, iğne'
chars  40 | bytes  48 | tokens  26 | 'Birinci satır\nİkinci satır\n\nüçüncü satır'
chars  14 | bytes  19 | tokens   8 | 'kar\xadşılaşmamız'
chars  14 | bytes  27 | tokens  20 | '👨\u200d👩\u200d👧 bir aile'
Round-trip checks passed.


### Encoding the Training and Held-Out Text

Encoding `train_text` must reproduce exactly the training sequence `ids`: applying merges in learned order replays the training loop. The held-out text gives the compression ratio on text the merges were not learned from.

In [19]:
start = time.perf_counter()
assert encode(train_text) == ids # encoding replays the training merges in their learned order
print('encode(train_text) == ids, seconds:', round(time.perf_counter() - start, 2))

val_tokens = list(val_text.encode("utf-8"))
val_ids = encode(val_text)
assert decode(val_ids) == val_text
print('val bytes:', len(val_tokens), '| val tokens:', len(val_ids))
print(f"train compression ratio: {len(tokens) / len(ids):.2f}X")
print(f"val compression ratio:   {len(val_tokens) / len(val_ids):.2f}X")

encode(train_text) == ids, seconds: 11.5


val bytes: 33468 | val tokens: 15554
train compression ratio: 2.22X
val compression ratio:   2.15X


# 4. GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit, birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz.

## 4. Forced Splits Using Regex Patterns (GPT Series)

Basic BPE merges whatever is frequent, including pairs that cross word boundaries. GPT-2 prevents this by first splitting the text with a **regex pattern** into chunks (words, numbers, punctuation, whitespace). BPE runs inside each chunk and the token lists are concatenated, so **no merge crosses a chunk boundary**.

The patterns use Unicode properties, `\p{L}` (any letter, including `ç`, `ı`, `İ`) and `\p{N}` (any number), and possessive quantifiers such as `?+`. They need the third-party `regex` package instead of Python's `re`.

In [20]:
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']


In [21]:
# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


### The GPT-4 `cl100k_base` Pattern

| Alternative | Matches |
|---|---|
| `'(?i:[sdmt]\|ll\|ve\|re)` | English contractions `'s 'd 'm 't 'll 've 're`, case-insensitive |
| `[^\r\n\p{L}\p{N}]?+\p{L}+` | a word: letters, with at most one leading non-letter (usually a space) |
| `\p{N}{1,3}` | numbers, at most three digits per chunk |
| ` ?[^\s\p{L}\p{N}]++[\r\n]*` | punctuation runs, optionally after one space |
| `\s*[\r\n]` | whitespace ending in a newline |
| `\s+(?!\S)` | whitespace, leaving the last space to attach to the next word |
| `\s+` | any remaining whitespace |

At each position the alternatives are tried from left to right. I use the pattern from Karpathy's minbpe exercise; `tiktoken` stores a slightly revised version, so the next cell checks whether both split my training text identically. A matching split only means the same chunks; it says nothing about merge rules or token IDs.

In [22]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
gpt4pat = re.compile(GPT4_SPLIT_PATTERN)

print(re.findall(gpt4pat, "Hello've world123 how's are you!!!?"))
print(re.findall(gpt4pat, "    hello world!!!"))
print(re.findall(gpt4pat, "Saat 12345'te, «Neden?» dedi.\n\n  Evet."))
print(re.findall(gpt4pat, "kar\xadşılaşmamız")) # the soft hyphen is not a letter

# _pat_str is a private tiktoken attribute and may change between versions
same_split = re.findall(enc._pat_str, train_text) == re.findall(gpt4pat, train_text)
print("identical to tiktoken's cl100k_base split on train_text:", same_split)
assert same_split, "Regex pre-tokenization differs from cl100k_base on the training corpus"

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']
['   ', ' hello', ' world', '!!!']
['Saat', ' ', '123', '45', "'t", 'e', ',', ' «', 'Neden', '?»', ' dedi', '.\n\n', ' ', ' Evet', '.']
['kar', '\xadşılaşmamız']
identical to tiktoken's cl100k_base split on train_text: True


### Split the Training Text into Chunks

Each chunk becomes its own list of byte IDs. Joining the chunks must give back the training text exactly; otherwise the pattern would have dropped characters.

In [23]:
text_chunks = re.findall(gpt4pat, train_text)
assert ''.join(text_chunks) == train_text
ids_chunks = [list(ch.encode("utf-8")) for ch in text_chunks]
assert sum(len(chunk_ids) for chunk_ids in ids_chunks) == len(tokens)

print('chunks:', len(text_chunks), '| unique chunks:', len(set(text_chunks)))
print(text_chunks[:25])
print('most common:', Counter(text_chunks).most_common(12))

chunks: 45892 | unique chunks: 13022
['\u2003Simdiye', ' kadar', ' tesadüf', ' ettiğim', ' insanlardan', ' bir', ' tanesi', ' benim', ' üzerimde', ' belki', ' en', ' büyük', ' tesiri', ' yapmıştır', '.', ' Aradan', ' aylar', ' geçtiği', ' halde', ' bir', ' türlü', ' bu', ' tesirden', ' kurtulamadım', '.']
most common: [(',', 2471), ('.', 2037), (' bir', 1399), ('\u2003', 1163), (' ve', 811), ('...', 476), (' bu', 429), ('.\n', 305), (' kadar', 265), (' de', 240), (' gibi', 232), (' için', 219)]


### Train BPE Inside the Chunks

`get_stats` gets an optional `counts` dictionary so pair counts can be **accumulated chunk by chunk**. Pairs are counted inside each chunk only; the chunks are never concatenated, so the last byte of one chunk and the first byte of the next never form a pair. The merge is then applied to every chunk separately. Corpus, `vocab_size`, and tie-breaking are the same as for the basic tokenizer.

In [24]:
def get_stats(ids, counts=None):
    # counts can be passed in to accumulate pair counts over many chunks
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

regex_merges = {} # (int, int) -> int
regex_merge_counts = {}
start = time.perf_counter()
for i in range(num_merges):
    stats = {}
    for chunk_ids in ids_chunks:
        get_stats(chunk_ids, stats) # pairs are counted inside each chunk only
    if not stats:
        break
    pair = max(stats, key=stats.get) # ties: the first pair encountered wins
    idx = 256 + i
    ids_chunks = [merge(chunk_ids, pair, idx) for chunk_ids in ids_chunks]
    regex_merges[pair] = idx
    regex_merge_counts[pair] = stats[pair]
regex_seconds = time.perf_counter() - start
print('learned merges:', len(regex_merges), '| training seconds:', round(regex_seconds, 2))

regex_vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in regex_merges.items():
    regex_vocab[idx] = regex_vocab[p0] + regex_vocab[p1]
assert len(regex_vocab) == 256 + len(regex_merges) == vocab_size

learned merges: 256 | training seconds: 18.0


### Chunk-Aware Encoding and Decoding

`encode_chunk` is the same loop as `encode`, applied to one chunk with an explicit merge table. `regex_encode` splits the text with the GPT-4 pattern, encodes every chunk independently, and concatenates the IDs. Decoding does not need the pattern; it only uses `regex_vocab`.

In [25]:
def encode_chunk(text_bytes, merges):
    # same loop as encode(), for one chunk and an explicit merge table
    ids = list(text_bytes)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        ids = merge(ids, pair, merges[pair])
    return ids

def regex_encode(text):
    # split with the GPT-4 pattern, then run BPE inside every chunk independently
    ids = []
    for chunk in re.findall(gpt4pat, text):
        ids.extend(encode_chunk(chunk.encode("utf-8"), regex_merges))
    return ids

def regex_decode(ids):
    text_bytes = b"".join(regex_vocab[idx] for idx in ids)
    return text_bytes.decode("utf-8", errors="replace")

for s in samples:
    assert regex_decode(regex_encode(s)) == s

regex_ids = regex_encode(train_text)
assert regex_ids == [idx for chunk_ids in ids_chunks for idx in chunk_ids] # replays the chunked training
regex_val_ids = regex_encode(val_text)
assert regex_decode(regex_val_ids) == val_text
print('Regex round-trip checks passed.')

Regex round-trip checks passed.


### Check the Chunk Boundaries

A token may begin with a space (`' ve'` is a valid GPT-4 chunk), so the check is not about which characters a token contains. It checks **byte positions**: every chunk boundary of the text must also be a token boundary, so no token's byte span reaches into a neighbouring chunk. The same measurement on the basic tokenizer counts how many chunk boundaries its tokens cross.

In [26]:
def boundary_ends(pieces):
    # byte offsets where each piece ends
    ends, pos = set(), 0
    for b in pieces:
        pos += len(b)
        ends.add(pos)
    return ends

def check_chunk_boundaries(text):
    chunks = re.findall(gpt4pat, text)
    chunk_ends = boundary_ends(ch.encode('utf-8') for ch in chunks)
    regex_ends = boundary_ends(regex_vocab[idx] for idx in regex_encode(text))
    basic_ends = boundary_ends(vocab[idx] for idx in encode(text))
    # encoding each chunk separately and concatenating gives the full encoder's output
    per_chunk = [idx for ch in chunks for idx in encode_chunk(ch.encode('utf-8'), regex_merges)]
    assert per_chunk == regex_encode(text)
    assert chunk_ends <= regex_ends # every chunk boundary is a regex-token boundary
    return len(chunk_ends), len(chunk_ends - basic_ends)

for name, s in [('val_text', val_text)] + [(repr(s[:20]), s) for s in samples if s]:
    n_bounds, crossed = check_chunk_boundaries(s)
    print(f"{name:24} chunk boundaries {n_bounds:6d} | crossed by regex BPE 0 | crossed by basic BPE {crossed}")
print('Chunk-boundary checks passed.')

val_text                 chunk boundaries   5093 | crossed by regex BPE 0 | crossed by basic BPE 2863
'hello world'            chunk boundaries      2 | crossed by regex BPE 0 | crossed by basic BPE 0
'Pijamalı hasta yağız'   chunk boundaries     12 | crossed by regex BPE 0 | crossed by basic BPE 7
'Pijamalı hasta yağız'   chunk boundaries      7 | crossed by regex BPE 0 | crossed by basic BPE 3
"İstanbul'da çay içti"   chunk boundaries     10 | crossed by regex BPE 0 | crossed by basic BPE 3
'ıiİI: IĞDIR, ılık, İ'   chunk boundaries     11 | crossed by regex BPE 0 | crossed by basic BPE 4
'Birinci satır\nİkinci'  chunk boundaries      8 | crossed by regex BPE 0 | crossed by basic BPE 2
'kar\xadşılaşmamız'      chunk boundaries      2 | crossed by regex BPE 0 | crossed by basic BPE 0
'👨\u200d👩\u200d👧 bir aile' chunk boundaries      3 | crossed by regex BPE 0 | crossed by basic BPE 1
Chunk-boundary checks passed.


### Compare the Basic and Regex Tokenizers

In [27]:
basic_list = list(merges.items())
regex_list = list(regex_merges.items())
print(f"{'rank':>4} | {'basic BPE':<14} {'count':>6} | {'regex BPE':<14} {'count':>6}")
for rank in range(20):
    (bp, bidx), (rp, ridx) = basic_list[rank], regex_list[rank]
    print(f"{rank+1:>4} | {render(vocab[bidx]):<14} {merge_counts[bp]:>6} | {render(regex_vocab[ridx]):<14} {regex_merge_counts[rp]:>6}")

rank | basic BPE       count | regex BPE       count
   1 | 'ı'             10406 | 'ı'             10406
   2 | ' b'             5084 | ' b'             5084
   3 | 'ü'              4425 | 'ü'              4425
   4 | 'ar'             3890 | 'ar'             3890
   5 | 'n '             3866 | 'ş'              3700
   6 | 'ş'              3700 | 'er'             3310
   7 | 'e '             3496 | 'an'             3164
   8 | 'er'             3310 | 'en'             2838
   9 | 'a '             2942 | 'ir'             2813
  10 | 'ir'             2813 | 'in'             2724
  11 | '. '             2602 | 'ğ'              2540
  12 | 'ğ'              2540 | ' d'             2501
  13 | 'i '             2500 | ' k'             2492
  14 | 'ç'              2302 | 'ç'              2302
  15 | 'ak'             2286 | 'ak'             2286
  16 | 'an'             2211 | 'ın'             2181
  17 | 'in'             2059 | ' g'             2151
  18 | 'de'             1987 | ' s'           

In [28]:
basic_tokens = {vocab[i] for i in range(256, vocab_size)}
regex_tokens = {regex_vocab[i] for i in range(256, vocab_size)}
print('learned tokens shared by both:', len(basic_tokens & regex_tokens))
print('only basic:', len(basic_tokens - regex_tokens), '| only regex:', len(regex_tokens - basic_tokens))

letter_then_space = re.compile(r"\p{L}\s")
for name, vb in [('basic', vocab), ('regex', regex_vocab)]:
    learned = [vb[i].decode('utf-8', errors='replace') for i in range(256, vocab_size)]
    print(f"{name}: tokens with a letter followed by whitespace: {sum(bool(letter_then_space.search(t)) for t in learned):3d}"
          f" | tokens starting with a space: {sum(t.startswith(' ') for t in learned):3d}")

longest_first = lambda tokens_: sorted(tokens_, key=lambda b: (-len(b), b))[:10] # sets have no fixed order
print('\nexamples only in basic:', ' '.join(render(b) for b in longest_first(basic_tokens - regex_tokens)))
print('examples only in regex:', ' '.join(render(b) for b in longest_first(regex_tokens - basic_tokens)))

learned tokens shared by both: 129
only basic: 127 | only regex: 127
basic: tokens with a letter followed by whitespace:  46 | tokens starting with a space:  21
regex: tokens with a letter followed by whitespace:   0 | tokens starting with a space:  94

examples only in basic: ':\n\u2003\u2003«' '\n\u2003\u2003«' '.\n\u2003\u2003' 'yordum. ' '\n\u2003\u2003' 'yordu. ' 'a bir ' 'i bir ' 'için ' 'ları '
examples only in regex: ' olduğ' 'ıyordu' ' düş' ' insan' ' için' ' kadar' ' sonra' ' zaman' ' çık' 'iyordu'


In [29]:
rows = [
    ('basic BPE', len(merges), basic_seconds, len(tokens), len(ids), len(val_tokens), len(val_ids)),
    ('regex BPE', len(regex_merges), regex_seconds, len(tokens), len(regex_ids), len(val_tokens), len(regex_val_ids)),
]
lines = ['| Tokenizer | Merges | Training seconds | Train bytes | Train tokens | Train ratio | Val bytes | Val tokens | Val ratio |',
         '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for name, m_, sec, tb, tt, vb, vt in rows:
    lines.append(f"| {name} | {m_} | {sec:.1f} | {tb:,} | {tt:,} | {tb/tt:.3f} | {vb:,} | {vt:,} | {vb/vt:.3f} |")
display(Markdown('\n'.join(lines)))

| Tokenizer | Merges | Training seconds | Train bytes | Train tokens | Train ratio | Val bytes | Val tokens | Val ratio |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| basic BPE | 256 | 10.7 | 304,535 | 137,020 | 2.223 | 33,468 | 15,554 | 2.152 |
| regex BPE | 256 | 18.0 | 304,535 | 137,765 | 2.211 | 33,468 | 15,430 | 2.169 |

### Turkish Apostrophes

In [30]:
for word in ["Ankara'dan", "İstanbul'da", "Türkiye'nin", "Ankara’dan", "İstanbul’da", " ANKARA'DAN"]:
    print(repr(word))
    print('   regex chunks:', re.findall(gpt4pat, word))
    print('   basic BPE   :', ' '.join(render(vocab[i]) for i in encode(word)))
    print('   regex BPE   :', ' '.join(render(regex_vocab[i]) for i in regex_encode(word)))
    print('   cl100k_base :', ' '.join(render(enc.decode_single_token_bytes(t)) for t in enc.encode(word)))

"Ankara'dan"
   regex chunks: ['Ankara', "'d", 'an']
   basic BPE   : 'A' 'n' 'kar' 'a' "'" 'dan'
   regex BPE   : 'A' 'n' 'k' 'ar' 'a' "'" 'd' 'an'
   cl100k_base : 'An' 'k' 'ara' "'d" 'an'
"İstanbul'da"
   regex chunks: ['İstanbul', "'d", 'a']
   basic BPE   : 'İ' 'st' 'an' 'b' 'ul' "'" 'd' 'a'
   regex BPE   : 'İ' 'st' 'an' 'b' 'ul' "'" 'd' 'a'
   cl100k_base : 'İ' 'stanbul' "'d" 'a'
"Türkiye'nin"
   regex chunks: ['Türkiye', "'nin"]
   basic BPE   : 'T' 'ür' 'k' 'iy' 'e' "'" 'n' 'in'
   regex BPE   : 'T' 'ür' 'k' 'iye' "'" 'n' 'in'
   cl100k_base : 'T' 'ür' 'kiye' "'n" 'in'
'Ankara’dan'
   regex chunks: ['Ankara', '’dan']
   basic BPE   : 'A' 'n' 'kar' 'a' '\xe2\x80' '\x99' 'dan'
   regex BPE   : 'A' 'n' 'k' 'ar' 'a' '\xe2\x80' '\x99' 'dan'
   cl100k_base : 'An' 'k' 'ara' '’d' 'an'
'İstanbul’da'
   regex chunks: ['İstanbul', '’da']
   basic BPE   : 'İ' 'st' 'an' 'b' 'ul' '\xe2\x80' '\x99' 'd' 'a'
   regex BPE   : 'İ' 'st' 'an' 'b' 'ul' '\xe2\x80' '\x99' 'da'
   cl100k_base : 'İ' 's

In [31]:
print("ASCII ' in train_text:", train_text.count("'"), "| typographic ’ (U+2019):", train_text.count('’'))
contraction = re.compile(r"'(?i:[sdmt]|ll|ve|re)")
hits = [ch for ch in text_chunks if contraction.fullmatch(ch)]
print('chunks produced by the contraction alternative:', Counter(hits))
for m_ in re.finditer(r"\p{L}+['’]\p{L}+", train_text):
    print(f"{m_.group()!r:22} -> {re.findall(gpt4pat, m_.group())}")

ASCII ' in train_text: 7 | typographic ’ (U+2019): 67
chunks produced by the contraction alternative: Counter()
'Zevako’lar'           -> ['Zevako', '’lar']
'Vern’ler'             -> ['Vern', '’ler']
'Düma’lar'             -> ['Düma', '’lar']
'Lökok’un'             -> ['Lökok', '’un']
'Turgenyef’in'         -> ['Turgenyef', '’in']
'Büridan’ın'           -> ['Büridan', '’ın']
'Kleopatra’dan'        -> ['Kleopatra', '’dan']
'Sarto’nun'            -> ['Sarto', '’nun']
'Madonna’sını'         -> ['Madonna', '’sını']
'Madonna’nın'          -> ['Madonna', '’nın']
'Sarto’nun'            -> ['Sarto', '’nun']
'Turgenyef’in'         -> ['Turgenyef', '’in']
'Storm’un'             -> ['Storm', '’un']
'Tiedemann’ın'         -> ['Tiedemann', '’ın']
'Tiedemann’ın'         -> ['Tiedemann', '’ın']
'Tiergarten’e'         -> ['Tiergarten', '’e']
'Döppke’nin'           -> ['Döppke', '’nin']
'Tiedemann’la'         -> ['Tiedemann', '’la']
'Tiedemann’dan'        -> ['Tiedemann', '’dan']
'Anlantik’te'         

### What the Regex Changes

**Merges.** Both tokenizers learn the same first four merges (`ı`, ` b`, `ü`, `ar`) and share 129 of their 256 learned tokens. The basic tokenizer's early `n `, `e `, `a `, `i `, `. `, and `, ` cannot exist in the regex tokenizer, because no chunk contains a letter or punctuation mark followed by a space. Those merges go to word beginnings that carry their leading space (` d`, ` k`, ` g`, ` s`, ` bir`) and to more letter pairs (`en`, `ın`, `im`). Over the whole vocabulary, 46 basic tokens contain a letter followed by whitespace and 0 regex tokens do, while 94 regex tokens begin with a space, against 21 basic ones. `ıyordu`, ` olduğ`, ` insan`, and ` sonra` exist only in the regex vocabulary; `yordum. ` and `a bir ` only in the basic one.

**Boundaries.** On the held-out text, basic BPE tokens cross 2,863 of the 5,093 chunk boundaries. Regex BPE tokens cross none, as the assertions require.

**Compression.** Forbidding cross-chunk merges costs almost nothing here: 2.223 (basic) vs. 2.211 (regex) on the training text and 2.152 vs. 2.169 on the held-out text. The regex tokenizer is slightly better on unseen text, but this is a single run with one vocabulary size.

### Turkish Apostrophes

The first alternative `'(?i:[sdmt]|ll|ve|re)` was written for English contractions such as *it's*, *I'd*, *I'm*, and *don't*. In Turkish, the apostrophe separates a proper noun from its suffixes, and many suffixes begin with these letters: *-dan/-da* (ablative, locative), *-tan/-ta*, *-sı*, *-m*. The regex tries this alternative first, so:

- `Ankara'dan` → `['Ankara', "'d", 'an']` and `İstanbul'da` → `['İstanbul', "'d", 'a']`: the suffix is cut after its first letter, and BPE can no longer form `dan` or `da` there.
- `Türkiye'nin` → `['Türkiye', "'nin"]`: `n` is not in the contraction list, so the second alternative keeps `'nin` as one chunk.
- Case-insensitivity also matches `'D` in `ANKARA'DAN`.

The typographic apostrophe `’` (U+2019) is not part of the contraction rule. `Ankara’dan` → `['Ankara', '’dan']` and `İstanbul’da` → `['İstanbul', '’da']` keep the whole suffix in one chunk. This is the usual case in my corpus: the training text has 67 `’` and only 7 ASCII `'`, and no ASCII apostrophe is followed by `s, d, m, t`, so the contraction alternative never fires on the training text. The suffixed words are mostly foreign names (`Maria’nın`, `Tiedemann’ın`, `Atlantik’te`).

**Regex chunks are not final tokens.** Inside the chunk `’dan`, my regex tokenizer gives `\xe2\x80`, `\x99`, `dan`: the bytes of `’` are split because only its prefix `\xe2\x80` (shared with the em space) became a learned token. `cl100k_base` turns `’dan` into `’d` + `an`, because it has a learned `’d` token, so even the typographic version is cut after `d`, this time by BPE rather than by the regex.

Apart from apostrophes, the pattern does nothing specific to Turkish: suffixes attached directly to a word, as in ` kurtulamadım`, stay inside one letter chunk, and BPE alone decides how to split them. A soft hyphen is not a letter, so it splits `kar\xadşılaşmamız` into two chunks.

# 5. Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?

## 5. Turkish vs. English Tokenization

The Turkish paragraph comes from the **held-out** 10% of the corpus, so neither of my tokenizers learned merges from it. The English version is a faithful translation prepared for this comparison.

**Counting conventions**, the same for both languages:

- **characters:** `len(s)`, Unicode code points
- **bytes:** `len(s.encode('utf-8'))`
- **words:** `len(s.split())`, whitespace-separated; punctuation stays attached, and `Ankara'dan` counts as one word
- **tokens:** length of the encoded ID list

**Tokens per word** = tokens / words. **Bytes per token** = bytes / tokens, the compression ratio of Task 2.

In [32]:
turkish_paragraph = "Dün gece bir saniye bile uyuyamadım. Yatakta arkaüstü yatarak hep trendeki çocuğu düşündüm. Vagonun sarsıntılariyle kımıldıyan başını görür gibi oluyordum. Bol saçlı bir çocuk başı... Ne gözlerinin, ne saçlarının rengini, hattâ ne de ismini biliyordum. Ona hiç dikkat etmemiştim. Yanıbaşımda, bir adım ötemde durduğu halde bir kere merakla yüzüne bakmamıştım. Ayrılırken elini bile sıkmamıştım. Hiçbir şey, aman Yarabbi, kendi kızıma dair hiçbir şey bilmiyordum. Kadın muhakkak ki birçok şeyler sezmişti... Niçin bana o kadar hain bakmıştı? Herhalde bir şeyler tahmin etmişti... Ve kızı alıp gitti... Şimdi yoldalar... Tekerlekler bir raydan bir raya atlarken kızımın uyuyan başı hafifçe sarsılıyor..."

english_paragraph = "Last night I could not sleep for even a second. Lying on my back in bed, I kept thinking about the child on the train. I almost seemed to see her head stirring with the jolts of the carriage. A child's head with thick hair... I did not know the color of her eyes or of her hair, not even her name. I had paid no attention to her. Although she had stood right beside me, one step away, I had not once looked at her face with curiosity. When we parted, I had not even shaken her hand. Nothing, dear God, I knew nothing about my own daughter. The woman had surely sensed many things... Why had she looked at me so maliciously? She must have guessed something... And she took the girl and left... Now they are on their way... As the wheels jump from one rail to the next, my daughter's sleeping head is gently shaking..."

assert turkish_paragraph in val_text and turkish_paragraph not in train_text

In [33]:
def describe(language, tokenizer, s, ids_):
    words = len(s.split())
    n_bytes = len(s.encode('utf-8'))
    return {'language': language, 'tokenizer': tokenizer, 'chars': len(s), 'bytes': n_bytes,
            'words': words, 'tokens': len(ids_), 'tokens_per_word': len(ids_) / words,
            'bytes_per_token': n_bytes / len(ids_)}

def show_rows(rows):
    lines = ['| Text | Tokenizer | Characters | Bytes | Words | Tokens | Tokens / word | Bytes / token |',
             '|---|---|---:|---:|---:|---:|---:|---:|']
    for r in rows:
        lines.append(f"| {r['language']} | {r['tokenizer']} | {r['chars']:,} | {r['bytes']:,} | {r['words']:,} | {r['tokens']:,} | {r['tokens_per_word']:.2f} | {r['bytes_per_token']:.2f} |")
    display(Markdown('\n'.join(lines)))

for s in [turkish_paragraph, english_paragraph]:
    assert enc.decode(enc.encode(s)) == s
    assert decode(encode(s)) == s and regex_decode(regex_encode(s)) == s

paragraph_rows = [
    describe('Turkish paragraph', 'cl100k_base', turkish_paragraph, enc.encode(turkish_paragraph)),
    describe('English paragraph', 'cl100k_base', english_paragraph, enc.encode(english_paragraph)),
    describe('Turkish paragraph', 'basic BPE (512)', turkish_paragraph, encode(turkish_paragraph)),
    describe('Turkish paragraph', 'regex BPE (512)', turkish_paragraph, regex_encode(turkish_paragraph)),
    describe('English paragraph', 'regex BPE (512)', english_paragraph, regex_encode(english_paragraph)),
]
show_rows(paragraph_rows)

| Text | Tokenizer | Characters | Bytes | Words | Tokens | Tokens / word | Bytes / token |
|---|---|---:|---:|---:|---:|---:|---:|
| Turkish paragraph | cl100k_base | 701 | 778 | 98 | 296 | 3.02 | 2.63 |
| English paragraph | cl100k_base | 816 | 816 | 159 | 188 | 1.18 | 4.34 |
| Turkish paragraph | basic BPE (512) | 701 | 778 | 98 | 343 | 3.50 | 2.27 |
| Turkish paragraph | regex BPE (512) | 701 | 778 | 98 | 343 | 3.50 | 2.27 |
| English paragraph | regex BPE (512) | 816 | 816 | 159 | 551 | 3.47 | 1.48 |

The same measurements on the whole held-out text separate a paragraph-specific effect from the corpus-level behaviour of each tokenizer.

In [34]:
show_rows([
    describe('Turkish val_text', 'cl100k_base', val_text, enc.encode(val_text)),
    describe('Turkish val_text', 'basic BPE (512)', val_text, val_ids),
    describe('Turkish val_text', 'regex BPE (512)', val_text, regex_val_ids),
])

| Text | Tokenizer | Characters | Bytes | Words | Tokens | Tokens / word | Bytes / token |
|---|---|---:|---:|---:|---:|---:|---:|
| Turkish val_text | cl100k_base | 30,281 | 33,468 | 4,020 | 12,463 | 3.10 | 2.69 |
| Turkish val_text | basic BPE (512) | 30,281 | 33,468 | 4,020 | 15,554 | 3.87 | 2.15 |
| Turkish val_text | regex BPE (512) | 30,281 | 33,468 | 4,020 | 15,430 | 3.84 | 2.17 |

In [35]:
def pieces(ids_, vb=None):
    if vb is None:
        return [enc.decode_single_token_bytes(t).decode('utf-8', errors='backslashreplace') for t in ids_]
    return [vb[i].decode('utf-8', errors='backslashreplace') for i in ids_]

strip = lambda w: w.strip('.,?!;:«»')
for label, paragraph in [('Turkish', turkish_paragraph), ('English', english_paragraph)]:
    print(f"--- longest {label} words ---")
    for w in sorted(set(map(strip, paragraph.split())), key=lambda w: (-len(w), w))[:6]: # ties: alphabetical
        line = f"{w:18} cl100k {len(enc.encode(w)):2d}: {pieces(enc.encode(w))}"
        if label == 'Turkish':
            line += f"\n{'':18} regex  {len(regex_encode(w)):2d}: {pieces(regex_encode(w), regex_vocab)}"
        print(line)

--- longest Turkish words ---
sarsıntılariyle    cl100k  7: ['s', 'ars', 'ı', 'nt', 'ıl', 'ari', 'yle']
                   regex   7: ['s', 'ar', 's', 'ın', 'tı', 'lar', 'iyle']
Tekerlekler        cl100k  4: ['T', 'eker', 'lek', 'ler']
                   regex   6: ['T', 'ek', 'er', 'l', 'ek', 'ler']
Yanıbaşımda        cl100k  7: ['Y', 'an', 'ı', 'ba', 'şı', 'md', 'a']
                   regex   7: ['Y', 'an', 'ı', 'b', 'aş', 'ım', 'da']
bakmamıştım        cl100k  6: ['bak', 'm', 'am', 'ış', 't', 'ım']
                   regex   6: ['b', 'ak', 'm', 'am', 'ış', 'tım']
bilmiyordum        cl100k  5: ['bil', 'mi', 'y', 'ord', 'um']
                   regex   5: ['b', 'il', 'm', 'iyordu', 'm']
sıkmamıştım        cl100k  7: ['s', 'ı', 'km', 'am', 'ış', 't', 'ım']
                   regex   6: ['s', 'ık', 'm', 'am', 'ış', 'tım']
--- longest English words ---
maliciously        cl100k  3: ['mal', 'icious', 'ly']
daughter's         cl100k  2: ['daughter', "'s"]
attention          cl100k  1: ['a

### What the Measurements Show

**Words and tokens.** The same content takes **98 Turkish words** and **159 English words**. With `cl100k_base`, the Turkish paragraph needs **296 tokens (3.02 tokens per word)** and the English paragraph **188 tokens (1.18 tokens per word)**. Turkish needs about 1.57 times as many tokens in total, although it has fewer characters (701 vs. 816).

**Why Turkish needs more tokens.** Several factors act together, and this experiment cannot separate them exactly:

- **Morphology:** a Turkish word joins a root and a chain of suffixes. `bakmamıştım` is *bak-ma-mış-tı-m*: look + negation + past perfect + first person, "I had not looked". Long words with many possible suffix combinations are less likely to exist as single vocabulary entries: `cl100k_base` splits `bakmamıştım` into 6 tokens.
- **Vocabulary coverage and training data:** a tokenizer's merges reflect the text it was trained on. Frequent English words such as `something` and `attention` are single `cl100k_base` tokens, while each of the six longest Turkish words takes 4 to 7 tokens. This suggests weaker Turkish coverage, but OpenAI has not published the tokenizer's training data, so its share of Turkish cannot be measured here.
- **Different ways of expressing the same content:** Turkish expresses with suffixes much of what English expresses with separate words (*I had not*, *on the train* for `trendeki`). The English version therefore has 159 words against 98, so the two languages' words are not comparable units. Translation choices such as *dear God* for *aman Yarabbi* also change the counts.
- **One paragraph:** this is a single translated literary paragraph. Other texts, genres, or translations would give different numbers. The held-out table checks the Turkish side on more text, but there is no parallel English text for it.

**Tokens per word is not enough.** Words are not equal units across languages, so tokens per word mixes tokenizer quality with the way each language groups meaning into words. For the same content, **total tokens** (or bytes per token) is the fairer cost: it decides how much of a context window the text fills and how many predictions a model makes. Neither number shows how well a model understands either language.

**Corpus and vocabulary size.** My tokenizers were trained on Turkish, yet both need **343 tokens** for the Turkish paragraph (3.50 tokens per word, 2.27 bytes per token), more than the 296 of `cl100k_base`. The whole held-out text gives the same ordering: 2.69 bytes per token for `cl100k_base` against 2.15 (basic) and 2.17 (regex). The comparison combines a corpus advantage for my tokenizers with a vocabulary of about 100k tokens against 512, and here the vocabulary size wins. The corpus effect is visible in the other direction: my Turkish-trained regex tokenizer needs 551 tokens for the English paragraph (1.48 bytes per token), against 188 for `cl100k_base`.

**My tokenizer is not GPT-4's tokenizer.** The regex tokenizer uses the same split pattern as `cl100k_base`, but its 256 merges and its token IDs were learned from my corpus. The same text produces different tokens and different IDs; only the pre-tokenization step is shared.

## Special Tokens

GPT-2's vocabulary has 50,257 tokens: 256 bytes, 50,000 merges, and 1 special token `<|endoftext|>` that separates documents during training. GPT-2's `encoder.json` and `vocab.bpe` files correspond to my `vocab` and `merges`. Special tokens are not produced by BPE; tiktoken inserts them only when they are explicitly allowed.

In [36]:
print('cl100k_base n_vocab:', enc.n_vocab)
print(enc.encode("<|endoftext|>", allowed_special="all"))
try:
    enc.encode("<|endoftext|>")
except ValueError as e:
    print('ValueError:', str(e).splitlines()[0])

cl100k_base n_vocab: 100277
[100257]
ValueError: Encountered text corresponding to disallowed special token '<|endoftext|>'.


## SentencePiece and vocab_size

**SentencePiece** is a library with several tokenization models (unigram, BPE, character, word); Llama 2 and Mistral use its BPE model. That model works on **code points** rather than bytes. **Byte fallback** is an option of the configuration, not a property of every SentencePiece tokenizer: with `byte_fallback=True`, as in the Llama 2 settings of the lecture, characters outside the vocabulary are encoded as UTF-8 byte tokens; without it, they map to the unknown token. tiktoken and my tokenizers run BPE on bytes from the start. I do not train SentencePiece here.

**vocab_size** is a trade-off. In the Week 8 GPT, it sets the rows of `token_embedding_table` and the outputs of `lm_head`. A larger vocabulary compresses text into fewer tokens, so the same `block_size` covers more text, but every token is rarer in the training data and those two layers grow. `cl100k_base` has about 100k tokens; my tokenizers have 512.

## Final Notes

- In UTF-8, each Turkish letter of my sentence takes two bytes and `👋` takes four. A byte-level base vocabulary of 256 IDs represents all of them.
- With 256 merges, basic BPE compresses the training text 2.22X and the held-out text 2.15X. The first merges rebuild the Turkish letters `ı`, `ü`, `ş`, `ğ`, `ç`, then frequent letter pairs and word endings followed by a space.
- `encode` must apply merges in learned order: `encode(train_text)` reproduces the training sequence exactly, and every test string round-trips, including Turkish letters, `İ ı i I`, newlines, a soft hyphen, emoji, and the empty string. Arbitrary IDs can produce incomplete UTF-8, which `errors="replace"` displays as `�`.
- GPT-4 regex pre-tokenization removes every token that crosses a chunk boundary at almost the same compression (2.21X train, 2.17X held-out). Its English contraction rule cuts ASCII-apostrophe suffixes such as `'dan` and `'da` after their first letter; my corpus mostly uses `’`, where the rule does not apply.
- Under `cl100k_base`, the Turkish paragraph takes 3.02 tokens per word against 1.18 for English, and 296 against 188 tokens in total.

**Limitations:** one corpus of 338,003 bytes, one vocabulary size, one paragraph pair whose English version was translated for this comparison, and single runs; training times depend on the machine. The optional exercises (reproducing `cl100k_base` exactly, training the Week 8 GPT on BPE tokens) are not part of this notebook.